# Verify the published segments dataset — phase 3

**Purpose.** Prove that `jember-segments-16k`, *as a fresh kernel actually mounts it*, is
the dataset that was reviewed. This notebook produces nothing and trusts nothing: it
imports even `scorer.py` from the mount rather than carrying its own copy.

**Why it is separate.** Cell 9 of `02-data-build` checks the files in `/kaggle/working` —
inside the kernel that just wrote them, with every variable still in memory. That is a
check on *construction*. It cannot catch a dataset published without `_meta/`, a truncated
upload, a path that shifts once Kaggle mounts it, or a FLAC that a different image refuses
to read. Those are failures of *consumption*, and only an independent kernel can see them.

Duplicating a producer causes drift; duplicating a *consumer* is the point — its
independence from the producer is exactly what is being tested.

**The three phases of any heavy notebook:**

| phase | what runs | output |
|---|---|---|
| 1 — feasibility | `02-data-build` with `WRITE_AUDIO = False` (and `DEBUG_N_RECORDINGS = 5` first) | a few MB, downloaded and reviewed |
| 2 — production | `02-data-build` with `WRITE_AUDIO = True`, then publish via **New Dataset** | ~630 MB, never downloaded |
| 3 — verification | **this notebook**, against the published dataset | a few hundred KB |

**How to run.**
1. Attach `jember-segments-16k`. Also attach the pass-2 output if you kept it — that gives
   this notebook the fingerprint baseline for check H.
2. Accelerator **None**, Internet **off**.
3. Run All. A couple of minutes: 6.6k FLAC headers plus 200 full decodes.
4. Output tab → Download all (it is tiny), unzip onto the repo root.

A red board here means **do not start notebook 03** — republish first.

## 1 — Setup and mount discovery

Finds the clip root by counting `.flac` per directory, so a rename of the dataset does not
break anything. The companion files are expected in `_meta/` next to the clips.

In [ ]:
# ============================================================
# CELL 1 - SETUP AND MOUNT DISCOVERY
# ============================================================
# This notebook trusts NOTHING from the kernel that built the dataset. It only
# sees what notebook 03 will see: a mounted, published Kaggle dataset. Every
# check below is re-derived from the mount, including the scorer, which is
# imported from the dataset rather than pasted in here.
import os, sys, re, csv, json, math, time, random, hashlib, collections
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import soundfile as sf
pd.set_option("display.width", 200)

SEED = 20260825
random.seed(SEED); np.random.seed(SEED)

FINDINGS = {}
def rec(section, **kw): FINDINGS.setdefault(section, {}).update(kw)
def head(t): print("\n" + "=" * 74); print(t); print("=" * 74)
def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return round(float(o), 4)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return round(o, 4)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

# ---- OUTPUT CONVENTION (copied verbatim, NOTEBOOK_ID changed) ----
NOTEBOOK_ID  = "02-verify"
WORKING      = Path("/kaggle/working")
OUT_DIR      = WORKING / "findings" / NOTEBOOK_ID
ARTIFACT_DIR = OUT_DIR / "_kaggle-artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
def out(fn):
    p = OUT_DIR / fn; p.parent.mkdir(parents=True, exist_ok=True); return p

_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT
class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False
LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("output dir :", OUT_DIR)

INPUT = Path("/kaggle/input")

head("FIND THE PUBLISHED SEGMENTS DATASET")
# Marker-based, like every other notebook here: the directory holding the most
# .flac files is the clip root, whatever the dataset ended up being called.
flac_by_dir = collections.Counter()
for p in INPUT.glob("**/*.flac"):
    flac_by_dir[p.parent] += 1
SEG_ROOT = flac_by_dir.most_common(1)[0][0] if flac_by_dir else None
print("clip root :", SEG_ROOT or "NOT FOUND")
print("clips here: %d   (across %d directories)"
      % (flac_by_dir[SEG_ROOT] if SEG_ROOT else 0, len(flac_by_dir)))

MANIFEST = None
for cand in ([SEG_ROOT / "_meta" / "manifest.csv"] if SEG_ROOT else []) + \
            sorted(INPUT.glob("**/manifest.csv")):
    if cand.exists():
        MANIFEST = cand; break
META_ROOT = MANIFEST.parent if MANIFEST is not None else None
print("manifest  :", MANIFEST or "NOT FOUND")

# The fingerprints the reviewed pass produced, recorded here by hand on
# 2026-08-25 from the phase-1 run and confirmed identical on its re-run. Check H
# compares the published dataset against these. Kept in the notebook rather than
# read from an attached digest, because a baseline that travels with the thing
# it is checking is not much of a baseline.
#
# manifest.csv is deliberately absent: it changed when the audio moved from one
# file per segment to one file per recording (new start_sample / end_sample
# columns, new path values). Its hash comes from the digest inside the dataset.
#
# Updated 2026-08-25: scorer.py, lexicon_v1.csv and text_audit.json got new
# hashes too, from the same-day normalizer fix (sentence-initial lowercasing
# + period stripping, then an idempotency bug in that fix). split.csv and
# dropped_segments.csv are untouched by text normalization, so they still
# anchor against the pre-fix values - if THOSE move, something drifted that
# should not have.
EXPECTED_FINGERPRINTS = {
    "split.csv":            "a7192227554da0ab",
    "lexicon_v1.csv":       "25dabca43dd2fcfc",
    "scorer.py":            "3260677fe544f6b0",
    "text_audit.json":      "dd6d1bd31b70b01b",
    "dropped_segments.csv": "407c22c08f7b4ded",
}

# A pass-2 digest, if one happens to be attached, overrides the constant above.
BASELINE = None
for cand in sorted(INPUT.glob("**/dataset_findings.json")):
    try:
        d = json.loads(cand.read_text(encoding="utf-8"))
    except Exception:
        continue
    if "fingerprints" in d:
        BASELINE = d; print("baseline  :", cand); break
if BASELINE is None:
    print("baseline  : NOT FOUND - the fingerprint check will report 'no baseline'")
    print("            (attach the pass-2 output, or paste the hashes by hand)")

# ==================================================================
# MANUAL OVERRIDE - only if a line above says NOT FOUND
# ==================================================================
# SEG_ROOT = Path("/kaggle/input/datasets/thoriqfarras/jember-segments-16k")
# MANIFEST = SEG_ROOT / "_meta" / "manifest.csv"
# META_ROOT = MANIFEST.parent

if SEG_ROOT is None or MANIFEST is None:
    raise RuntimeError("attach the jember-segments-16k dataset, or fill the "
                       "MANUAL OVERRIDE block above and re-run this cell")

rec("env", notebook=NOTEBOOK_ID, seg_root=str(SEG_ROOT), manifest=str(MANIFEST),
    meta_root=str(META_ROOT), has_baseline=bool(BASELINE),
    python=sys.version.split()[0], pandas=pd.__version__)

## 2 — The checks

Eight groups, ordered so the cheapest failure surfaces first. Group D reads **every** clip
header rather than a sample — headers are cheap, and it turns "we looked at twenty" into a
statement about the whole dataset. Group E decodes 200 clips for real, because a correct
header can still sit in front of digital silence.

Group H is the one that gives the three-phase workflow its meaning: it compares the sha256
of every companion file against the fingerprints recorded in the reviewed pass.

The closing block measures how small a WER difference this dev set can actually resolve —
worth knowing before any GPU time is spent chasing a gap that is inside the noise.

In [ ]:
# ============================================================
# CELL 2 - THE CHECKS
# ============================================================
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:200]))
    print("  [%s] %-56s %s" % ("PASS" if cond else "FAIL", name, detail))
    return bool(cond)

head("A - IS THE PACKAGE COMPLETE?")
expected_meta = ["manifest.csv", "split.csv", "lexicon_v1.csv", "scorer.py",
                 "text_audit.json", "dropped_segments.csv"]
present = sorted(p.name for p in META_ROOT.iterdir() if p.is_file()) if META_ROOT else []
missing = [n for n in expected_meta if n not in present]
check("_meta/ carries every companion file", not missing, "missing: %s" % missing if missing else present)

head("B - DOES THE SHIPPED SCORER STILL WORK IN A FRESH KERNEL?")
# Imported from the MOUNT. If notebook 03 can import it, so can this check;
# if it cannot, better to find out here than three cells into a training run.
if str(META_ROOT) not in sys.path:
    sys.path.insert(0, str(META_ROOT))
import scorer
print("  scorer version:", getattr(scorer, "VERSION", "?"), "from", scorer.__file__)
ok_scorer, sub = scorer.self_test(verbose=False)
check("scorer.self_test() passes from the mount", ok_scorer,
      "%d/%d" % (sum(1 for _, c, _ in sub if c), len(sub)))
for nm, c, det in sub:
    if not c:
        print("      FAILED:", nm, det)

head("C - MANIFEST AND RECORDINGS AGREE")
M = pd.read_csv(MANIFEST, dtype={"audio_filename": str, "seg_id": str}, keep_default_na=False)
print("  manifest rows: %d   columns: %d" % (len(M), M.shape[1]))
print("  audio is stored one FLAC per RECORDING; segments are sample offsets.")
flacs = {p.stem: p for p in SEG_ROOT.glob("*.flac")}
check("one flac per recording", len(flacs) == M["audio_filename"].nunique(),
      "%d flac / %d recordings" % (len(flacs), M["audio_filename"].nunique()))
absent = sorted(set(M["audio_filename"]) - set(flacs))
check("every manifest row points at a recording that exists", not absent,
      "%d missing, e.g. %s" % (len(absent), absent[:5]))
orphan = sorted(set(flacs) - set(M["audio_filename"]))
check("no orphan recording without manifest rows", not orphan,
      "%d orphans, e.g. %s" % (len(orphan), orphan[:5]))
check("seg_id unique", M["seg_id"].is_unique, "%d unique" % M["seg_id"].nunique())
for col in ("start_sample", "end_sample", "path"):
    check("manifest carries %s" % col, col in M.columns)

head("D - EVERY RECORDING HEADER, NOT A SAMPLE")
# 208 headers cost under a second, so there is no reason to sample. The
# important check is the last one: no segment may address a sample past the end
# of its recording, which is the failure a truncated upload would produce.
t0 = time.time()
bad_sr, bad_ch, zero_len, unreadable = [], [], [], []
tot_frames = 0
frames = {}
for fid, p in sorted(flacs.items()):
    try:
        info = sf.info(str(p))
    except Exception as e:
        unreadable.append((fid, type(e).__name__)); continue
    frames[fid] = info.frames
    tot_frames += info.frames
    if info.samplerate != 16000: bad_sr.append(fid)
    if info.channels != 1:       bad_ch.append(fid)
    if info.frames == 0:         zero_len.append(fid)
print("  read %d headers in %.1f s" % (len(flacs), time.time() - t0))
check("every recording is readable", not unreadable, unreadable[:5])
check("every recording is 16 kHz", not bad_sr, "%d bad, e.g. %s" % (len(bad_sr), bad_sr[:5]))
check("every recording is mono", not bad_ch, "%d bad, e.g. %s" % (len(bad_ch), bad_ch[:5]))
check("no zero-length recording", not zero_len, "%d empty, e.g. %s" % (len(zero_len), zero_len[:5]))
over = [(r["seg_id"], int(r["end_sample"]), frames.get(r["audio_filename"], -1))
        for _, r in M.iterrows()
        if int(r["end_sample"]) > frames.get(r["audio_filename"], -1)]
check("no segment addresses a sample past its recording", not over,
      "%d over, e.g. %s" % (len(over), over[:3]))
h_disk = tot_frames / 16000.0 / 3600.0
h_seg = float(M["dur_s"].astype(float).sum()) / 3600.0
print("  %.3f h of recordings on disk, %.3f h addressed by segments" % (h_disk, h_seg))
check("segments address between 90 and 100 %% of the audio",
      0.90 <= h_seg / max(h_disk, 1e-9) <= 1.001, "%.1f %%" % (100 * h_seg / max(h_disk, 1e-9)))
check("every segment is under 30 s", bool(M["dur_s"].astype(float).max() < 30.0),
      "longest %.2f s" % float(M["dur_s"].astype(float).max()))

head("E - READ SEGMENTS THE WAY NOTEBOOK 03 WILL")
# Headers can be right while the audio behind them is silence, and an offset
# can be in range yet point at the wrong place. This decodes real segments
# through the exact call the training dataloader will make.
rows = M.sample(n=min(200, len(M)), random_state=SEED)
silent, clipped, nan_bad, wrong_len, peaks, rms_all = [], [], [], [], [], []
for _, r in rows.iterrows():
    a, b = int(r["start_sample"]), int(r["end_sample"])
    y, sr = sf.read(str(SEG_ROOT / ("%s.flac" % r["audio_filename"])),
                    start=a, stop=b, dtype="float32")
    if y.ndim > 1: y = y.mean(axis=1)
    if len(y) != b - a or sr != 16000:
        wrong_len.append(r["seg_id"]); continue
    if not np.isfinite(y).all(): nan_bad.append(r["seg_id"]); continue
    pk = float(np.abs(y).max()) if y.size else 0.0
    rmsv = float(np.sqrt((y.astype(np.float64) ** 2).mean())) if y.size else 0.0
    peaks.append(pk); rms_all.append(rmsv)
    if rmsv < 1e-5: silent.append(r["seg_id"])
    if pk >= 0.999 and float((np.abs(y) >= 0.999).mean()) > 0.01: clipped.append(r["seg_id"])
check("200 segments read back at exactly the requested length", not wrong_len, wrong_len[:5])
check("no NaN or inf in the decoded audio", not nan_bad, nan_bad[:5])
check("no digitally silent segment in the sample", not silent,
      "%d silent, e.g. %s" % (len(silent), silent[:5]))
print("  sample peak   : min %.3f  median %.3f  max %.3f"
      % (min(peaks), float(np.median(peaks)), max(peaks)))
print("  sample rms dB : min %.1f  median %.1f  max %.1f"
      % (20 * math.log10(min(rms_all) + 1e-12), 20 * math.log10(float(np.median(rms_all)) + 1e-12),
         20 * math.log10(max(rms_all) + 1e-12)))

head("E2 - CLIPPING, MEASURED OVER THE WHOLE CORPUS")
# A sample of 200 can tell you clipping exists; it cannot tell you WHY, and the
# why decides what to do. Concentrated in a few recordings = those sessions were
# recorded too hot, which is a property of the source and nothing to fix here.
# Spread evenly across many recordings = the int16 conversion in notebook 02 is
# overshooting, which is a real bug and must be fixed before anyone trains.
# Decoding all 208 recordings costs about a minute, so measure instead of guess.
t0 = time.time()
CLIP = 0.999
seg_clip_frac = {}
rec_clip = {}
for fid, p in sorted(flacs.items()):
    y, sr = sf.read(str(p), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    hot = (np.abs(y) >= CLIP).astype(np.int32)
    rec_clip[fid] = float(hot.mean())
    cum = np.concatenate([[0], np.cumsum(hot)])
    for _, r in M[M["audio_filename"] == fid].iterrows():
        a, b = int(r["start_sample"]), min(int(r["end_sample"]), len(y))
        if b > a:
            seg_clip_frac[r["seg_id"]] = (cum[b] - cum[a]) / float(b - a)
print("  scanned %d recordings in %.1f s" % (len(flacs), time.time() - t0))

BAD = 0.01                                   # >1 % of samples pinned to full scale
bad_segs = sorted((s for s, f in seg_clip_frac.items() if f > BAD),
                  key=lambda s: -seg_clip_frac[s])
bad_recs = sorted((f for f in rec_clip if rec_clip[f] > 0.001), key=lambda f: -rec_clip[f])
n_seg = max(len(seg_clip_frac), 1)
pct_seg = 100.0 * len(bad_segs) / n_seg
print("  segments with >1 %% of samples clipped : %d of %d (%.2f %%)"
      % (len(bad_segs), n_seg, pct_seg))
print("  recordings with any clipping at all   : %d of %d" % (len(bad_recs), len(flacs)))
if bad_recs:
    print("  worst recordings:")
    for f in bad_recs[:8]:
        sp = M.loc[M["audio_filename"] == f, "split"].iloc[0]
        n_here = sum(1 for s in bad_segs if s.rsplit("_", 1)[0] == f)
        print("    rec %-5s %6.2f %% of samples clipped   %-5s  %d bad segments"
              % (f, 100 * rec_clip[f], sp, n_here))
dev_bad = [s for s in bad_segs
           if M.loc[M["seg_id"] == s, "split"].iloc[0] == "dev"]
print("  of those, %d are in dev (%.2f %% of dev segments)"
      % (len(dev_bad), 100.0 * len(dev_bad) / max(int((M["split"] == "dev").sum()), 1)))

# The verdict distinguishes the two causes described above.
concentrated = len(bad_recs) <= max(5, 0.05 * len(flacs))
check("clipping is not widespread (would mean a conversion bug)",
      pct_seg < 5.0 and concentrated,
      "%.2f %% of segments, in %d recordings" % (pct_seg, len(bad_recs)))
if bad_segs:
    print("")
    print("  >>> Clipping lives in the source mp3s, not in the conversion: it is")
    print("  >>> confined to a few sessions that were recorded too hot. Republishing")
    print("  >>> cannot fix it. Recorded here so notebook 07 can check whether these")
    print("  >>> segments carry more than their share of the errors.")

head("F - SPLIT INTEGRITY, RE-DERIVED FROM THE MOUNT")
SP = pd.read_csv(META_ROOT / "split.csv", dtype=str, keep_default_na=False)
both = set(M.loc[M["split"] == "train", "audio_filename"]) & \
       set(M.loc[M["split"] == "dev", "audio_filename"])
check("no recording in both train and dev", not both, sorted(both)[:5])
check("split.csv agrees with the manifest",
      dict(zip(SP["audio_filename"], SP["split"])) ==
      dict(zip(M["audio_filename"], M["split"])), "%d recordings" % len(SP))
dev_h = float(M.loc[M["split"] == "dev", "dur_s"].astype(float).sum()) / 3600
print("  train %.2f h / dev %.2f h  (dev = %.1f %% of duration)"
      % (h_seg - dev_h, dev_h, 100 * dev_h / max(h_seg, 1e-9)))

head("G - TEXT AND SCORER INVARIANTS ON THE SHIPPED REFERENCES")
refs = M["text_official"].astype(str).tolist()
check("official_normalize is idempotent on the shipped text",
      all(scorer.official_normalize(t) == t for t in refs), "%d rows" % len(refs))
check("no empty reference", all(t.strip() for t in refs))
w0 = scorer.wer(refs, refs)["wer"]
w1 = scorer.wer(refs, [""] * len(refs))["wer"]
check("wer(refs, refs) == 0.0", w0 == 0.0, "%.6f" % w0)
check("wer(refs, empty) == 1.0", w1 == 1.0, "%.6f" % w1)
LEX = scorer.load_lexicon(META_ROOT / "lexicon_v1.csv")
check("lexicon loads and covers the references", len(LEX) > 0,
      "%d entries, %.1f %% of ref tokens found"
      % (len(LEX), 100.0 * sum(1 for t in refs for w in t.split()
                               if scorer.lex_key(w) in LEX) / max(sum(len(t.split()) for t in refs), 1)))

head("H - IS THIS THE DATASET THAT WAS REVIEWED?")
# The fingerprints from the analysis pass. This is the check that makes the
# whole three-phase workflow mean something: it proves the artifacts inside the
# published dataset are byte-for-byte the ones whose numbers were agreed.
fp_now = {}
for name in expected_meta:
    p = META_ROOT / name
    if p.exists():
        fp_now[name] = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
fp_want = dict(EXPECTED_FINGERPRINTS)
fp_want.update((BASELINE or {}).get("fingerprints", {}))   # digest wins if attached
if fp_want:
    # Only files with a known baseline are judged. A file the baseline does not
    # cover is printed for the record, not failed - absence of a reference is
    # not evidence of corruption.
    diff = {k: (fp_want[k], fp_now.get(k)) for k in sorted(fp_want)
            if k in fp_now and fp_now[k] != fp_want[k]}
    for k in sorted(fp_now):
        if k in fp_want:
            print("  %-24s %s %s" % (k, fp_now[k], "== baseline" if fp_now[k] == fp_want[k]
                                     else "!= baseline (%s)" % fp_want[k]))
        else:
            print("  %-24s %s  (no baseline for this file)" % (k, fp_now[k]))
    absent_fp = [k for k in fp_want if k not in fp_now]
    check("fingerprints match the reviewed pass", not diff and not absent_fp,
          ("mismatched: %s" % diff) if diff else
          (("missing from the dataset: %s" % absent_fp) if absent_fp else
           "%d of %d files verified" % (len(fp_want) - len(absent_fp), len(fp_now))))
else:
    for k in sorted(fp_now):
        print("  %-24s %s" % (k, fp_now[k]))
    print("  no baseline attached - compare these against the pass-1/pass-2 output by eye")
    CHECKS.append(dict(check="fingerprints match the reviewed pass", passed=True,
                       detail="SKIPPED - no baseline digest attached"))

head("DEV-SET RESOLUTION - HOW SMALL A DIFFERENCE IS STILL REAL?")
# Answering this before any GPU runs is what keeps notebooks 04-06 from
# celebrating noise. The bootstrap resamples utterances, not tokens.
dev = M.loc[M["split"] == "dev", "text_official"].astype(str).tolist()
rnd = random.Random(SEED)
def _degrade(t, p):
    keep = []
    for w in t.split():
        u = rnd.random()
        if u < p * 0.5:   keep.append(w.lower())
        elif u < p * 0.8: continue
        elif u < p:       keep.extend([w, "e"])
        else:             keep.append(w)
    return " ".join(keep)
for level in (0.30, 0.50):
    hyp = [_degrade(t, level) for t in dev]
    ci = scorer.wer_ci(dev, hyp, n_boot=400, seed=SEED)
    print("  simulated WER %.3f  ->  95%% CI [%.3f, %.3f]   +/- %.3f (%.2f points)"
          % (ci["wer"], ci["lo"], ci["hi"], ci["half_width"], 100 * ci["half_width"]))
print("")
print("  dev utterances: %d   dev reference tokens: %d"
      % (len(dev), sum(len(t.split()) for t in dev)))
print("  >>> Treat any WER gap smaller than roughly twice that half-width as")
print("  >>> unproven from a single run. Use scorer.compare_systems() for the")
print("  >>> paired test - it is far more sensitive than two overlapping CIs.")

n_pass = sum(1 for c in CHECKS if c["passed"])
print("")
print("=" * 74)
print("%d/%d checks passed" % (n_pass, len(CHECKS)))
print(">>> DATASET VERIFIED. Notebook 03 may mount it." if n_pass == len(CHECKS)
      else ">>> DO NOT USE THIS DATASET YET - fix the FAIL rows and republish.")
print("=" * 74)

rec("verify", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS),
    verified=bool(n_pass == len(CHECKS)), n_recordings=len(flacs), manifest_rows=len(M),
    clipping=dict(segments_over_1pct=len(bad_segs), segments_pct=round(pct_seg, 3),
                  recordings_affected=len(bad_recs), dev_segments=len(dev_bad),
                  worst=[[f, round(100 * rec_clip[f], 3)] for f in bad_recs[:10]],
                  worst_segments=[[s, round(100 * seg_clip_frac[s], 2)] for s in bad_segs[:20]]),
    hours_on_disk=round(h_disk, 4), hours_addressed=round(h_seg, 4),
    dev_hours=round(dev_h, 4), dev_utterances=len(dev),
    dev_ref_tokens=int(sum(len(t.split()) for t in dev)),
    fingerprints=fp_now, sample_decoded=len(rows))

## 3 — Digest

Paste the marked block into the chat.

In [ ]:
# ============================================================
# CELL 3 - DIGEST
# ============================================================
EXPECTED = ["env", "verify"]
missing_sections = [k for k in EXPECTED if k not in FINDINGS]
FINDINGS["_meta"] = dict(
    notebook="mozilla-javanese dataset verification (phase 3)",
    notebook_id=NOTEBOOK_ID,
    generated_for="handoff to Claude - do not edit by hand",
    complete=(not missing_sections), missing_sections=missing_sections)

blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False,
                  separators=(",", ":"), sort_keys=True)
print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
print(blob)
print("===== END CLAUDE-HANDOFF-JSON =====")
print("")
print("size: %.1f KB - paste the whole block into the chat" % (len(blob) / 1024))

out("verify_report.json").write_text(
    json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True),
    encoding="utf-8")
print("saved ->", out("verify_report.json"))
sys.stdout.flush()
print("saved ->", LOG_PATH)

try:
    moved = []
    for q in WORKING.iterdir():
        if q.is_file():
            q.replace(ARTIFACT_DIR / q.name); moved.append(q.name)
    if moved:
        print("swept into _kaggle-artifacts:", moved)
except Exception as e:
    print("sweep skipped:", e)

print("")
print("This whole output is a few hundred KB. Download all, unzip onto the repo")
print("root, and findings/%s/ lands where it belongs." % NOTEBOOK_ID)